# GNNExplainer 

## Local Explainability

In [17]:
from torch_geometric.explain import Explainer, GNNExplainer
import pandas as pd
import os
import torch

from models import GoldenTransformer
from graph_construction import build_graph
from resources import OUTPUT_TABLES_PATH, DATA_PATH, DEVICE, MODELS_PATH
from data_preparation import prepare_dataset, get_loss_weights


In [18]:
best_transformer_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "best_transformer_df.csv"))

x, y, pos_combined, pos_spatial, pos_temporal = prepare_dataset(DATA_PATH, seasonal_features=True)
class_weights = get_loss_weights(y = y)

log_area_t = x.to(DEVICE) if x.dim() == 2 else x.unsqueeze(1).to(DEVICE)
x_full = torch.cat([
    log_area_t, 
    pos_spatial.to(DEVICE), 
    pos_temporal.to(DEVICE)
], dim=1)

There are 10034 natural fires (56.44 %)
There are 7744 human made fires (43.56 %)


In [19]:
best_gnn_row = best_transformer_df.sort_values(by='F1', ascending=False).iloc[0]
best_k, best_h = int(best_gnn_row['K']), int(best_gnn_row['Heads'])

data = build_graph('multirelational', best_k, pos_spatial, pos_temporal, pos_combined, x_full, y)
data = data.to(DEVICE)
in_dim = data.x.size(1)       
e_dim = data.edge_attr.size(1)

model_gnn = GoldenTransformer(
    input_dim=data.x.size(1), 
    hidden_dim=16, 
    edge_dim=data.edge_attr.size(1), 
    num_heads=best_h
).to(DEVICE)
model_gnn.load_state_dict(torch.load(os.path.join(MODELS_PATH, "best_gnn_weights.pth")))

<All keys matched successfully>

In [20]:

explainer = Explainer(
    model = model_gnn,
    algorithm = GNNExplainer(epochs=200),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(
        mode="multiclass_classification",
        task_level="node",
        return_type="raw"
    )
)

In [79]:
import networkx as nx
import matplotlib.pyplot as plt
import torch
import pandas as pd

def create_importance_plots(explanation, target_node_idx):
    # 1. Prediction & Setup
    with torch.no_grad():
        logits = model_gnn(data.x, data.edge_index, data.edge_attr)
        prediction_idx = logits[target_node_idx].argmax().item()
        pred_label = "Natural" if prediction_idx == 0 else "Human"

    _, (ax_graph, ax_node_feat) = plt.subplots(1, 2, figsize=(18, 8), dpi=300)

    local_target = explanation.index if hasattr(explanation, 'index') else target_node_idx
    if isinstance(local_target, torch.Tensor):
        local_target = local_target.item()

    edge_index = explanation.edge_index
    edge_mask = explanation.edge_mask
    
    # 2. Identify Top 10 High-Importance Edges
    k_edges = min(10, edge_mask.numel())
    top_k_values, _ = torch.topk(edge_mask, k_edges)
    threshold = top_k_values.min().item()

    G = nx.Graph()
    node_labels = {local_target: f"PRED:\n{pred_label}"}

    # Add Top 10 important edges
    for i in range(edge_index.shape[1]):
        u, v = edge_index[0, i].item(), edge_index[1, i].item()
        weight = edge_mask[i].item()
        if weight >= threshold:
            is_temporal = explanation.edge_attr[i, 3].item() == 1
            etype = "Temporal" if is_temporal else "Spatial"
            G.add_edge(u, v, weight=weight, type=etype, important=True)
            if u not in node_labels: node_labels[u] = ""
            if v not in node_labels: node_labels[v] = ""

    if local_target not in G:
        G.add_node(local_target)

    # 3. Connect all islands
    components = list(nx.connected_components(G))
    target_comp = next(c for c in components if local_target in c)
    other_comps = [c for c in components if local_target not in c]

    for comp in other_comps:
        best_bridge_weight = -1
        best_bridge_edge = None
        for i in range(edge_index.shape[1]):
            u, v = edge_index[0, i].item(), edge_index[1, i].item()
            is_bridge = (u in comp and v in target_comp) or (v in comp and u in target_comp)
            if is_bridge and edge_mask[i].item() > best_bridge_weight:
                best_bridge_weight = edge_mask[i].item()
                best_bridge_edge = (u, v, i)
        
        if best_bridge_edge:
            u, v, idx = best_bridge_edge
            is_temporal = explanation.edge_attr[idx, 3].item() == 1
            etype = "Temporal" if is_temporal else "Spatial"
            G.add_edge(u, v, weight=best_bridge_weight, type=etype, important=False)

    # 4. Center the Target & Prevent Overlap
    # Lock the target node at (0,0)
    fixed_pos = {local_target: (0, 0)}
    # Use a stronger 'k' to push nodes away from each other
    pos = nx.spring_layout(G, k=1.5, pos=fixed_pos, fixed=[local_target], seed=42)

    # 5. Drawing
    node_colors = ['#e74c3c' if n == local_target else '#f39c12' for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=1200, node_color=node_colors, edgecolors='black', ax=ax_graph)

    for u, v, d in G.edges(data=True):
        if d.get('important', False):
            color = '#c0392b' if d['type'] == "Temporal" else '#2980b9'
            width = max(2.5, d['weight'] * 12)
            nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=width, edge_color=color, alpha=0.8, ax=ax_graph)
        else:
            # Bridges connecting components to target
            nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=1.5, edge_color='black', style='--', alpha=0.5, ax=ax_graph)

    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=10, font_weight='bold', ax=ax_graph)
    
    imp_labels = {(u, v): d['type'] for u, v, d in G.edges(data=True) if d.get('important', False)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=imp_labels, font_size=8, ax=ax_graph)

    ax_graph.set_title(f"Target-Centered Explanation (Node {target_node_idx})", fontsize=14, fontweight='bold')
    ax_graph.axis('off')

    # 6. Node feature importance
    node_feat_names = ['Log-Area', 'Season_Sin', 'Season_Cos', 'Latitude', 'Longitude', 'Days Since Start']
    node_mask = explanation.node_mask
    importances = node_mask.mean(dim=0).cpu().numpy() if node_mask.dim() == 2 else node_mask.squeeze().cpu().numpy()

    if len(importances) == len(node_feat_names):
        node_df = pd.DataFrame({'Feature': node_feat_names, 'Importance': importances}).sort_values(by='Importance')
        ax_node_feat.barh(node_df['Feature'], node_df['Importance'], color='#27ae60')
        ax_node_feat.set_title("Influential Node Features", fontsize=14, fontweight='bold')
        ax_node_feat.set_xlabel("Importance Score")
        ax_node_feat.grid(axis='x', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

In [80]:
import matplotlib.pyplot as plt

def batch_explain_and_plot(indices, label_name):
    print(f"--- Generating Plots for {label_name} Fires ---")
    for idx in indices:
        node_idx = idx.item()
        
        explanation = explainer(data.x, data.edge_index, edge_attr=data.edge_attr, index=node_idx)

        print(f"Explaining Node {node_idx} (True Label: {label_name})")
        create_importance_plots(explanation, node_idx)

        plt.show() 


In [ ]:
# Filter for 5 Correct Naturals and 5 Correct Humans
with torch.no_grad():
    logits = model_gnn(data.x, data.edge_index, data.edge_attr)
    preds = logits.argmax(dim=-1)

correct_mask = (preds == data.y) & data.test_mask
correct_indices = correct_mask.nonzero(as_tuple=False).view(-1)

target_natural = correct_indices[data.y[correct_indices] == 0][:5]
target_human = correct_indices[data.y[correct_indices] == 1][:5]

batch_explain_and_plot(target_natural, "Correctly Predicted Natural")
batch_explain_and_plot(target_human, "Correctly Predicted Human")

## Global Explainability

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def generate_global_explainability(explainer, data, mask, num_samples=30):
    indices = mask.nonzero(as_tuple=False).view(-1)
    if len(indices) > num_samples:
        perm = torch.randperm(len(indices))[:num_samples]
        indices = indices[perm]

    all_node_feat_masks = []
    # track the total importance weight assigned to each edge type
    total_spatial_importance = 0
    total_temporal_importance = 0

    print(f"Aggregating {len(indices)} local explanations...")

    for i, node_idx in enumerate(indices):
        node_idx = node_idx.item()
        explanation = explainer(data.x, data.edge_index, index=node_idx, edge_attr=data.edge_attr)
        
        # 1. Node Feature Importance
        node_mask = explanation.node_mask
        all_node_feat_masks.append(node_mask.mean(dim=0).cpu().numpy() if node_mask.dim() == 2 else node_mask.squeeze().cpu().numpy())
            
        # 2. Edge Type Importance
        edge_mask = explanation.edge_mask
        is_temporal = explanation.edge_attr[:, 3] == 1
        is_spatial = explanation.edge_attr[:, 3] == 0
        
        # Sum the learned weights for each category
        total_temporal_importance += edge_mask[is_temporal].sum().item()
        total_spatial_importance += edge_mask[is_spatial].sum().item()

    # Average Node Features
    global_node_importance = np.mean(all_node_feat_masks, axis=0)
    node_feat_names = ['Log-Area', 'Season_Sin', 'Season_Cos', 'Latitude', 'Longitude', 'Days Since Start']
    node_df = pd.DataFrame({'Feature': node_feat_names, 'Importance': global_node_importance}).sort_values('Importance')

    # Prepare Binary Edge Comparison
    edge_data = pd.DataFrame({
        'Type': ['Spatial', 'Temporal'],
        'Cumulative Importance': [total_spatial_importance, total_temporal_importance]
    })

    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Node Plot
    ax1.barh(node_df['Feature'], node_df['Importance'], color='seagreen')
    ax1.set_title(f"Global Node Feature Importance (n={num_samples})", fontweight='bold')
    
    # Edge Plot (Binary)
    ax2.bar(edge_data['Type'], edge_data['Cumulative Importance'], color=['royalblue', 'indianred'])
    ax2.set_title("Global Edge Type Preference", fontweight='bold')
    ax2.set_ylabel("Total Explainer Weight")
    
    plt.tight_layout()
    plt.show()

    return node_df, edge_data

In [ ]:
# Run with your desired sample size
global_node_df, global_edge_df = generate_global_explainability(explainer, data, data.test_mask, num_samples=100)